# Notebook 7: analysis for the report (CPU only, no GPU, no Internet)
**Attach Datasets: `nli-data`, and every Dataset holding prediction files:** Notebook 1 output (LR + hypothesis-only LR), Notebook 2 output (baseline DistilBERT),
Notebook 3 output (Llama LoRA: `preds_llama1b_lora_*`), and the Notebook 6 outputs (`kd`, `poe`, `ls`, `kd_generic`).
Files are found by name anywhere under `/kaggle/input`, so any number of Datasets works. Missing models are skipped, so you can run this early and again later.

Produces in `/kaggle/working/analysis/`: main results table, hard/easy table, significance tests, MNLI per-genre table, accuracy-by-bucket table,
gap-closure table, compute table, 4 figures, and `errors_sample.csv` (rows to hand-tag for the qualitative error analysis).

In [1]:
import glob, json, os, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = os.environ.get("NLI_ROOT", "/kaggle/input")
def find(name, required=True):
    h = glob.glob(f"{BASE}/**/{name}", recursive=True)
    if required: assert h, f"not found: {name}"
    return h[0] if h else None

ROOT = Path(find("nli_common.py")).parent; sys.path.insert(0, str(ROOT)); DATA = str(ROOT / "data")
from nli_common import LABELS, load_split, mcnemar, evaluate
OUT = Path(os.environ.get("NLI_OUT", "/kaggle/working/analysis")); OUT.mkdir(parents=True, exist_ok=True)

EVALS = ["snli_val", "snli_test", "mnli_m", "mnli_mm"]
TEST_SETS = ["snli_test", "mnli_m", "mnli_mm"]
SEEDS = [42, 43, 44]
MODELS = {   # display name -> (file prefix template, seeds)
    "LR": ("lr", [None]), "LR hyp-only": ("lr_hyponly", [None]),
    "DistilBERT baseline": ("distilbert_s{seed}", SEEDS),
    "DB + distillation (kd)": ("distilbert_kd_s{seed}", SEEDS),
    "DB + product of experts": ("distilbert_poe_s{seed}", SEEDS),
    "DB + label smoothing (control)": ("distilbert_ls_s{seed}", SEEDS),
    "DB + distillation + generic": ("distilbert_kd_generic_s{seed}", SEEDS),
    "DB + distillation + extra SNLI (control)": ("distilbert_kd_extra_s{seed}", SEEDS),
    "Llama-1B LoRA": ("llama1b_lora", [None]),
}
BASEN, LLAMA = "DistilBERT baseline", "Llama-1B LoRA"
D = {es: load_split(es, DATA) for es in EVALS}

RUNS = {}     # RUNS[model][es] = list of prediction DataFrames (one per seed)
for name, (tpl, seeds) in MODELS.items():
    for es in EVALS:
        dfs = []
        for s in seeds:
            p = find(f"preds_{tpl.format(seed=s)}_{es}.csv", required=False)
            if p:
                d = pd.read_csv(p); assert (d["id"].to_numpy() == D[es]["id"].to_numpy()).all(), f"{name} {es}: ids not aligned"
                dfs.append(d)
        if dfs: RUNS.setdefault(name, {})[es] = dfs
present = [m for m in MODELS if m in RUNS]
print("models found:", {m: len(RUNS[m].get("snli_test", [])) for m in present})
print("missing:", [m for m in MODELS if m not in RUNS])
OK = {m: {es: np.stack([(d["pred"] == d["label"]).to_numpy() for d in RUNS[m][es]]) for es in RUNS[m]} for m in present}   # (seeds, N)
fmt = lambda v: f"{100 * np.mean(v):.2f}" + (f" ± {100 * np.std(v, ddof=1):.2f}" if len(v) > 1 else "")

models found: {'LR': 1, 'LR hyp-only': 1, 'DistilBERT baseline': 3, 'DB + distillation (kd)': 3, 'DB + product of experts': 3, 'DB + label smoothing (control)': 3, 'DB + distillation + generic': 3, 'DB + distillation + extra SNLI (control)': 3, 'Llama-1B LoRA': 1}
missing: []


## 1. Main results (accuracy and macro-F1, mean ± std over seeds)

In [2]:
rows = []
for m in present:
    r = {"model": m, "seeds": len(RUNS[m].get("snli_test", []))}
    for es in EVALS:
        if es in RUNS[m]:
            r[es + "_acc"] = fmt([x.mean() for x in OK[m][es]])
            r[es + "_f1"] = fmt([evaluate(d["label"], d["pred"])["f1_macro"] for d in RUNS[m][es]])
    rows.append(r)
main = pd.DataFrame(rows); main.to_csv(OUT / "main_results.csv", index=False)
print(main[["model", "seeds", "snli_val_acc", "snli_test_acc", "mnli_m_acc", "mnli_mm_acc"]].to_string(index=False))

                                   model  seeds snli_val_acc snli_test_acc   mnli_m_acc  mnli_mm_acc
                                      LR      1        74.68         74.56        49.18        49.09
                             LR hyp-only      1        64.51         64.34        40.82        41.55
                     DistilBERT baseline      3 85.94 ± 0.13  85.82 ± 0.05 62.79 ± 0.30 65.05 ± 0.43
                  DB + distillation (kd)      3 87.15 ± 0.25  87.12 ± 0.12 65.81 ± 0.32 67.47 ± 0.71
                 DB + product of experts      3 84.72 ± 0.32  84.34 ± 0.41 59.92 ± 1.02 61.77 ± 1.11
          DB + label smoothing (control)      3 86.03 ± 0.27  85.89 ± 0.12 63.14 ± 1.09 65.06 ± 1.30
             DB + distillation + generic      3 87.13 ± 0.24  87.35 ± 0.27 70.63 ± 0.73 72.08 ± 0.37
DB + distillation + extra SNLI (control)      3 87.98 ± 0.10  87.93 ± 0.05 67.12 ± 0.46 69.00 ± 0.25
                           Llama-1B LoRA      1        91.98         91.24        82.69    

## 2. Hard vs easy SNLI-test examples (hard = rows the hypothesis-only LR gets wrong)

In [3]:
HARD = None
if "LR hyp-only" in RUNS and "snli_test" in RUNS["LR hyp-only"]:
    h = RUNS["LR hyp-only"]["snli_test"][0]; HARD = (h["pred"] != h["label"]).to_numpy()
    print(f"hard subset: {HARD.sum()} of {len(HARD)} test rows ({100 * HARD.mean():.1f}%)")
    he = [{"model": m, "overall": fmt(OK[m]["snli_test"].mean(1)), "hard": fmt(OK[m]["snli_test"][:, HARD].mean(1)),
           "easy": fmt(OK[m]["snli_test"][:, ~HARD].mean(1))} for m in present if m != "LR hyp-only"]
    he = pd.DataFrame(he); he.to_csv(OUT / "hard_easy.csv", index=False); print(he.to_string(index=False))

hard subset: 3503 of 9824 test rows (35.7%)
                                   model      overall         hard         easy
                                      LR        74.56        50.16        88.09
                     DistilBERT baseline 85.82 ± 0.05 73.52 ± 0.12 92.63 ± 0.02
                  DB + distillation (kd) 87.12 ± 0.12 75.35 ± 0.23 93.64 ± 0.10
                 DB + product of experts 84.34 ± 0.41 77.08 ± 0.28 88.37 ± 0.51
          DB + label smoothing (control) 85.89 ± 0.12 73.61 ± 0.45 92.70 ± 0.23
             DB + distillation + generic 87.35 ± 0.27 75.75 ± 0.28 93.78 ± 0.27
DB + distillation + extra SNLI (control) 87.93 ± 0.05 76.88 ± 0.35 94.05 ± 0.19
                           Llama-1B LoRA        91.24        82.96        95.82


## 3. Is each difference real? (vs the DistilBERT baseline)
- 95% bootstrap CI of the accuracy difference, resampling test examples, using per-example correctness averaged over seeds.
- McNemar's exact test per seed (same seed vs same seed); `sig` = number of seeds with p < 0.05.

In [4]:
rng = np.random.default_rng(0); sig_rows = []
KDN, KDG, KDE = "DB + distillation (kd)", "DB + distillation + generic", "DB + distillation + extra SNLI (control)"
comparisons = [(m, BASEN) for m in present if m not in (BASEN, "LR", "LR hyp-only")]
comparisons += [(KDG, KDN)] * (KDG in RUNS and KDN in RUNS) + [(KDG, KDE)] * (KDG in RUNS and KDE in RUNS) + [(KDE, KDN)] * (KDE in RUNS and KDN in RUNS)
for m, ref in comparisons:
    if ref not in RUNS: continue
    for es in TEST_SETS:
        if es not in RUNS[m] or es not in RUNS[ref]: continue
        a, b = OK[m][es].mean(0), OK[ref][es].mean(0); d = a - b
        idx = rng.integers(0, len(d), (1000, len(d))); boot = d[idx].mean(1)
        k = min(len(RUNS[m][es]), len(RUNS[ref][es])); ps, wins = [], []
        for i in range(k):
            bb, cc, p = mcnemar(RUNS[ref][es][i]["pred"], RUNS[m][es][i]["pred"], D[es]["label"]); ps.append(p); wins.append(cc - bb)
        sig_rows.append({"model": m, "vs": ref, "eval_set": es, "diff_pts": round(100 * d.mean(), 2),
                         "ci95": f"[{100 * np.percentile(boot, 2.5):+.2f}, {100 * np.percentile(boot, 97.5):+.2f}]",
                         "seeds_sig(p<.05)": f"{sum(p < .05 for p in ps)}/{k}", "mcnemar_p_per_seed": [f"{p:.3g}" for p in ps]})
sig = pd.DataFrame(sig_rows); sig.to_csv(OUT / "significance_vs_baseline.csv", index=False)
print(sig.to_string(index=False) if len(sig) else "no variants to compare yet")

                                   model                                       vs  eval_set  diff_pts             ci95 seeds_sig(p<.05)             mcnemar_p_per_seed
                  DB + distillation (kd)                      DistilBERT baseline snli_test      1.30   [+1.00, +1.61]              3/3  [1.5e-06, 1.68e-09, 4.99e-08]
                  DB + distillation (kd)                      DistilBERT baseline    mnli_m      3.02   [+2.55, +3.50]              3/3  [2.87e-18, 7.61e-17, 2.1e-16]
                  DB + distillation (kd)                      DistilBERT baseline   mnli_mm      2.42   [+1.94, +2.87]              3/3 [1.83e-15, 2.23e-13, 6.92e-08]
                 DB + product of experts                      DistilBERT baseline snli_test     -1.48   [-1.89, -1.10]              3/3 [4.12e-08, 0.000693, 8.59e-11]
                 DB + product of experts                      DistilBERT baseline    mnli_m     -2.87   [-3.53, -2.19]              3/3 [1.57e-14, 5.24e-06, 2.26e-11

## 4. How much of the baseline-to-Llama gap does each variant close?

In [5]:
gap = []
if BASEN in RUNS and LLAMA in RUNS:
    for m in present:
        if m in (BASEN, LLAMA, "LR", "LR hyp-only"): continue
        r = {"model": m}
        for es in TEST_SETS:
            if es in OK[m]:
                b, l, v = OK[BASEN][es].mean(), OK[LLAMA][es].mean(), OK[m][es].mean()
                r[es + "_closed_%"] = round(100 * (v - b) / (l - b), 1)
        gap.append(r)
    gap = pd.DataFrame(gap); gap.to_csv(OUT / "gap_closure.csv", index=False); print(gap.to_string(index=False))

                                   model  snli_test_closed_%  mnli_m_closed_%  mnli_mm_closed_%
                  DB + distillation (kd)                24.0             15.2              13.6
                 DB + product of experts               -27.2            -14.4             -18.5
          DB + label smoothing (control)                 1.4              1.8               0.0
             DB + distillation + generic                28.3             39.4              39.6
DB + distillation + extra SNLI (control)                38.9             21.7              22.2


## 5. MNLI accuracy by genre

In [6]:
GEN = {}
for es in ["mnli_m", "mnli_mm"]:
    g = D[es]["genre"].to_numpy(); tab = {}
    for m in present:
        if es in OK[m] and m != "LR hyp-only":
            c = OK[m][es].mean(0); tab[m] = {gn: 100 * c[g == gn].mean() for gn in sorted(set(g))}
    GEN[es] = pd.DataFrame(tab).round(1); GEN[es].to_csv(OUT / f"genre_{es}.csv"); print(f"\n{es}:\n", GEN[es].to_string())


mnli_m:
               LR  DistilBERT baseline  DB + distillation (kd)  DB + product of experts  DB + label smoothing (control)  DB + distillation + generic  DB + distillation + extra SNLI (control)  Llama-1B LoRA
fiction     48.3                 61.7                    65.7                     59.8                            62.0                         69.3                                      66.3           83.1
government  51.9                 68.9                    71.7                     65.1                            69.0                         76.3                                      72.9           84.5
slate       46.8                 59.1                    61.9                     56.5                            59.7                         67.7                                      63.9           79.9
telephone   51.1                 58.1                    61.5                     55.4                            58.7                         67.6                       

## 6. Accuracy by example property (negation, word overlap, lengths)

In [7]:
TOK = re.compile(r"[a-z0-9']+")
NEG = {"not", "no", "nobody", "nothing", "never", "none", "nowhere", "neither", "nor", "cannot"}
def feats(df):
    P = [set(TOK.findall(x.lower())) for x in df["premise"]]; Hs = [TOK.findall(x.lower()) for x in df["hypothesis"]]
    return pd.DataFrame({"hyp_negation": ["yes" if any(w in NEG or w.endswith("n't") for w in h) else "no" for h in Hs],
                         "overlap": [len(set(h) & p) / max(1, len(set(h))) for h, p in zip(Hs, P)],
                         "hyp_len": [len(h) for h in Hs], "prem_len": [len(p.split()) for p in df["premise"]]})
bucket_rows = []
for es in TEST_SETS:
    F = feats(D[es])
    for col in ["overlap", "hyp_len", "prem_len"]:
        F[col + "_bucket"] = pd.qcut(F[col], 3, duplicates="drop").astype(str)
    for col in ["hyp_negation", "overlap_bucket", "hyp_len_bucket", "prem_len_bucket"]:
        for val in sorted(v for v in F[col].unique() if v != "nan"):
            msk = (F[col] == val).to_numpy(); r = {"eval_set": es, "property": col, "value": val, "n": int(msk.sum())}
            for m in present:
                if es in OK[m] and m != "LR hyp-only": r[m] = round(100 * OK[m][es][:, msk].mean(), 1)
            bucket_rows.append(r)
buckets = pd.DataFrame(bucket_rows); buckets.to_csv(OUT / "accuracy_by_property.csv", index=False)
print(buckets[buckets.eval_set == "snli_test"].to_string(index=False))

 eval_set        property           value    n   LR  DistilBERT baseline  DB + distillation (kd)  DB + product of experts  DB + label smoothing (control)  DB + distillation + generic  DB + distillation + extra SNLI (control)  Llama-1B LoRA
snli_test    hyp_negation              no 9619 74.6                 85.8                    87.1                     84.4                            85.9                         87.3                                      87.9           91.2
snli_test    hyp_negation             yes  205 72.2                 85.4                    87.8                     82.8                            85.0                         88.9                                      87.3           92.7
snli_test  overlap_bucket (-0.001, 0.333] 3577 73.4                 85.2                    86.3                     82.8                            85.0                         86.7                                      87.4           90.9
snli_test  overlap_bucket  (0.333, 0.568

## 7. Compute table

In [8]:
def extra(prefix_tpl, seeds):
    out = []
    for s in seeds:
        p = find(f"results_{prefix_tpl.format(seed=s)}_snli_test.json", required=False)
        if p:
            j = json.load(open(p)); e = j["extra"]; n = j["metrics"]["n"]
            eps = e.get("inference_examples_per_s") or (n / e["inference_s"] if e.get("inference_s") else np.nan)
            out.append({"params_M": (e.get("n_params") or e.get("n_total_params") or np.nan) / 1e6, "train_min": e.get("train_time_s", np.nan) / 60,
                        "peak_gpu_GB": e.get("peak_gpu_mem_gb", np.nan), "infer_ex_per_s": eps})
    return pd.DataFrame(out).mean().round(1).to_dict() if out else None
comp = [{"model": m, **x} for m in present if m != "LR hyp-only" for x in [extra(*MODELS[m])] if x]
comp = pd.DataFrame(comp); comp.to_csv(OUT / "compute.csv", index=False); print(comp.to_string(index=False))

                                   model  params_M  train_min  peak_gpu_GB  infer_ex_per_s
                                      LR       2.0        1.7          NaN         14170.9
                     DistilBERT baseline      67.0        9.2          1.9          3004.4
                  DB + distillation (kd)      67.0        9.4          1.9          3309.4
                 DB + product of experts      67.0        9.5          1.9          3231.0
          DB + label smoothing (control)      67.0        9.5          1.9          3252.1
             DB + distillation + generic      67.0       19.4          2.1          3249.3
DB + distillation + extra SNLI (control)      67.0       14.0          1.9          3254.1
                           Llama-1B LoRA    1247.1       51.7          8.0            79.2


## 8. Figures

In [9]:
show = [m for m in present if m != "LR hyp-only"]
fig, ax = plt.subplots(figsize=(11, 4.5)); w = 0.8 / len(show)
for i, m in enumerate(show):
    ax.bar(np.arange(len(TEST_SETS)) + i * w, [100 * OK[m][es].mean() if es in OK[m] else 0 for es in TEST_SETS], w, label=m)
ax.set_xticks(np.arange(len(TEST_SETS)) + 0.4 - w / 2); ax.set_xticklabels(["SNLI test", "MNLI-matched", "MNLI-mismatched"])
ax.set_ylabel("accuracy (%)"); ax.set_ylim(40, 100); ax.legend(fontsize=7, ncol=2); ax.set_title("Trained on SNLI 100K, evaluated on SNLI and MNLI")
fig.tight_layout(); fig.savefig(OUT / "fig1_accuracy.png", dpi=160); plt.close(fig)

if "mnli_m" in GEN:
    ax = GEN["mnli_m"].plot.bar(figsize=(11, 4.5), width=0.85); ax.set_ylabel("accuracy (%)"); ax.set_title("MNLI-matched accuracy by genre")
    ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.savefig(OUT / "fig2_genre.png", dpi=160); plt.close()

if HARD is not None:
    fig, ax = plt.subplots(figsize=(9, 4)); w = 0.8 / len(show)
    for i, m in enumerate(show):
        ax.bar(np.arange(2) + i * w, [100 * OK[m]["snli_test"][:, HARD].mean(), 100 * OK[m]["snli_test"][:, ~HARD].mean()], w, label=m)
    ax.set_xticks(np.arange(2) + 0.4 - w / 2); ax.set_xticklabels(["hard (hyp-only wrong)", "easy (hyp-only right)"]); ax.set_ylim(30, 100)
    ax.set_ylabel("accuracy (%)"); ax.legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(OUT / "fig3_hard_easy.png", dpi=160); plt.close(fig)

cm_models = [m for m in [BASEN, "DB + distillation (kd)", "DB + product of experts", LLAMA] if m in RUNS and "mnli_m" in RUNS[m]]
if cm_models:
    fig, axs = plt.subplots(1, len(cm_models), figsize=(3.6 * len(cm_models), 3.6))
    for a, m in zip(np.atleast_1d(axs), cm_models):
        cm = np.array(evaluate(RUNS[m]["mnli_m"][0]["label"], RUNS[m]["mnli_m"][0]["pred"])["confusion_matrix"], float); cm /= cm.sum(1, keepdims=True)
        a.imshow(cm, vmin=0, vmax=1, cmap="Blues"); a.set_title(m, fontsize=8); a.set_xticks(range(3)); a.set_yticks(range(3))
        a.set_xticklabels(["ent", "neu", "con"], fontsize=7); a.set_yticklabels(["ent", "neu", "con"], fontsize=7); a.set_xlabel("predicted", fontsize=7)
        for i in range(3):
            for j in range(3): a.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > .5 else "black")
    fig.suptitle("MNLI-matched, row-normalised confusion (seed 42 / single run)", fontsize=9); fig.tight_layout(); fig.savefig(OUT / "fig4_confusion_mnli_m.png", dpi=160); plt.close(fig)

## 9. Error samples to hand-tag (qualitative error analysis)
Categories use seed-42 predictions. Tag about 50 rows yourself (negation, over-inference, lexical-overlap trap, label noise, ambiguous...).

In [10]:
samples = []; rs = np.random.default_rng(1)
def pred(m, es, i=0): return RUNS[m][es][min(i, len(RUNS[m][es]) - 1)]["pred"].to_numpy()
KD = "DB + distillation (kd)"
for es in ["snli_test", "mnli_m"]:
    if not (BASEN in RUNS and es in RUNS[BASEN]): continue
    y = D[es]["label"].to_numpy(); b = pred(BASEN, es); cats = {}
    if LLAMA in RUNS and es in RUNS[LLAMA]:
        l = pred(LLAMA, es); cats["baseline wrong, Llama right"] = (b != y) & (l == y); cats["both wrong (noise candidates)"] = (b != y) & (l != y)
    if KD in RUNS and es in RUNS[KD]:
        k = pred(KD, es); cats["kd fixes baseline error"] = (b != y) & (k == y); cats["kd breaks baseline success"] = (b == y) & (k != y)
    for cat, msk in cats.items():
        idx = np.where(msk)[0]; idx = rs.choice(idx, min(15, len(idx)), replace=False) if len(idx) else idx
        for i in idx:
            r = {"eval_set": es, "category": cat, "id": int(D[es]["id"].iloc[i]), "premise": D[es]["premise"].iloc[i],
                 "hypothesis": D[es]["hypothesis"].iloc[i], "gold": LABELS[y[i]], "baseline": LABELS[b[i]], "tag": ""}
            if LLAMA in RUNS and es in RUNS[LLAMA]: r["llama"] = LABELS[l[i]]
            if KD in RUNS and es in RUNS[KD]: r["kd"] = LABELS[k[i]]
            samples.append(r)
pd.DataFrame(samples).to_csv(OUT / "errors_sample.csv", index=False)
print(f"{len(samples)} sample rows saved. Files:", sorted(os.listdir(OUT)))

120 sample rows saved. Files: ['accuracy_by_property.csv', 'compute.csv', 'errors_sample.csv', 'fig1_accuracy.png', 'fig2_genre.png', 'fig3_hard_easy.png', 'fig4_confusion_mnli_m.png', 'gap_closure.csv', 'genre_mnli_m.csv', 'genre_mnli_mm.csv', 'hard_easy.csv', 'main_results.csv', 'significance_vs_baseline.csv']
